In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 벡터DB : Chroma vs Pinecone
- Chroma : 인메모리DB, 로컬메모리 DB
- Pinecone : 클라우드 vector DB
    (Pinecone console에 api key 생성 -> .env (PINECONE_API_KEY)

# 0. 패키지 설치

In [ ]:
%pip install -q pinecone-client langchain-pinecone

# 1. Knowledge Base 구성을 위한 데이터 생성

In [1]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./tax_docs/with_table.docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)
document_list = loader.load_and_split(text_splitter=text_splitter)

In [2]:
len(document_list)

225

In [3]:
# embedding: upstage embedding-query
# https://python.langchain.com/v0.2/docs/integrations/text_embedding/upstage/#usage
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(
    model='solar-embedding-1-large'
#     model='embedding-query'
)

In [4]:
%%time
# pincone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()
# 데이터를 처음 업로드할 때
index_name = 'tax-index-table'
database = PineconeVectorStore.from_documents(
    documents=document_list,
    embedding=embedding,
    index_name=index_name
)
# 업로드한 벡터DB 가져올 때
# database = PineconeVectorStore(
#     embedding=embedding,   # 질문을 임베딩하여 유사도 검색
#     index_name=index_name
# )

C:\Users\Admin\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU times: total: 19.8 s
Wall time: 1min 4s


# 2. 답변 생성을 위한 Retrieval

In [5]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
# retriever = database.as_retriever(
#     #search_kwargs={'k':4}
# )
# retriever.invoke(query)

# 3. 제공되는 prompt를 활용하여 답변 생성

In [6]:
from langchain import hub
prompt = hub.pull('rlm/rag-prompt')

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [7]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={'prompt':prompt}
)

In [8]:
ai_message = qa_chain.invoke({'query':query})
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5천만원인 직장인의 소득세는 약 624만원입니다. 이는 5,000만원 초과 8,800만원 이하 구간의 세율(24%)에 해당하며, 과세표준 계산 후 세액이 도출됩니다. 정확한 세액은 공제액 등을 고려해야 하니 참고용으로 이해하시기 바랍니다.'}